In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
import seaborn as sns
import colorcet
import itertools
from scipy import interpolate, stats
from scipy.optimize import curve_fit
import statsmodels.formula.api as smf

import mathieu as mh

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/experiments/moby_complementation/'
#fig_path = f'{base_dir}fig/'
fig_path = '/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

In [ ]:
experiments = pd.read_csv(f'{base_dir}script/experiments.csv').set_index('dataset')
experiments.drop('batch0', inplace=True)
strains_info = pd.read_csv(f'{base_dir}script/strains_info.csv').set_index('strain')

In [ ]:
# Generate the plate template
for batch in experiments.index:
    hu = experiments.loc[batch, 'HU']
    plasmids_row = pd.read_csv(f'{base_dir}script/plasmids_row_{batch}.csv', index_col=0).iloc[:, 0]
    strains_col = pd.read_csv(f'{base_dir}script/strains_col_{batch}.csv', index_col=0).iloc[:, 0]
    rows = np.repeat(['A','B','C','D','E','F','G','H','I','J','K','L','M','N','O','P'], 24)
    cols = np.tile(np.arange(1, 25), 16)
    
    samples_info = pd.DataFrame([rows, cols], index=['row', 'col']).T
    samples_info['well'] = samples_info[['row', 'col']].apply(lambda x: ''.join(x.astype(str).values), axis=1)
    samples_info['strain'] = strains_col.loc[samples_info['col']].values
    samples_info['plasmid'] = plasmids_row.loc[samples_info['row']].values
    if batch == 'batch0':
        samples_info['HU'] = np.where(samples_info['col'].isin(np.arange(3, 24, 2)), hu, 0)
    else:
        samples_info['HU'] = np.where(samples_info['col'].isin(np.arange(2, 25, 2)), hu, 0)
    samples_info['rep'] = np.where(samples_info['row'].isin(['A','C','E','G','I','K','M','O']), 1, 2)
    
    #samples_info.to_csv(f'{base_dir}script/samples_{batch}.csv', index=None)

In [ ]:
collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx')
collection.index = collection['ID'].values

strain_order = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]
strain_order = collection.loc[strain_order].sort_values(by='Y\' element').index
strain_color = dict(zip(strain_order, colorcet.glasbey_category10))
#strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in collection.loc[strain_order, 'Y\' element']/150]))
strain_yprime_color = dict(zip(strain_order, [cm.viridis(i) for i in np.linspace(0,1,10)]))
hu_high_color = dict(zip([0]+[2**i for i in range(4,11)], [cm.magma(i) for i in np.linspace(0,1,8)]))

hu_color = {0:'k', 256:'orange', 128:'pink'}

# Analysis of the first batch with empty plasmid control

In [ ]:
GC_DAT = []
GC_samples = {}

for ds, (dat_file, sm_file, temp, duration, HU, blank_correct) in experiments.iterrows():

    dat_path = f'{base_dir}data/{dat_file}'
    sm_path = f'{base_dir}script/{sm_file}'
        
    sm, gc_dat = mh.parse_gc(dat_path, sm_path, ds, skiprows=11, blank_correct=blank_correct, Tend=duration)
    
    GC_DAT.append(gc_dat)
    GC_samples[ds] = sm
GC_DAT = pd.concat(GC_DAT).reset_index(drop=True)
GC_DAT = GC_DAT.loc[(~GC_DAT['OD600'].isna()) & (GC_DAT['strain']!='blank')]

for (ds, w), df in GC_DAT.groupby(['dataset', 'well']):
    zero = df.set_index('time').loc[0, 'OD600_blank']
    GC_DAT.loc[df.index, 'log_OD600'] = np.log(df['OD600_blank']/zero)

In [ ]:
# per-well inspection

ds_color_dict = dict(zip(GC_DAT['dataset'].unique(), colorcet.glasbey_bw))

for well, df in GC_DAT.groupby('well'):

    fig, ax = plt.subplots(figsize=[5,5])

    for ds, df1 in df.groupby('dataset'):
        ax.plot(df1['time'], df1['OD600'], c=ds_color_dict[ds], label=ds)
        
    ax.set_xticks(np.arange(0, 30, 4), minor=False)
    ax.set_xticks(np.arange(0, 30, 1), minor=True)
    ax.grid(which='major', lw=1)
    ax.grid(which='minor', lw=0.3)

    ax.legend(loc=2)
    
    ax.set_ylim(0, 2.6)
    ax.set_xlabel('hrs')
    ax.set_ylabel('OD$_{600}$')
    ax.set_title(well)

    plt.tight_layout()
    plt.savefig(f'{fig_path}per_well/{well}.png', dpi=300)
    #plt.show()
    plt.close()

# Standard AUC analysis

In [ ]:
GC_AUC = []
GC_AUC_MEAN = []

for ds, gc_dat in GC_DAT.groupby('dataset'):
    
    gc_samples = GC_samples[ds]
    gc_auc, gc_auc_mean = mh.compute_gc_auc(gc_dat, GC_samples[ds], strains_info, ds, od='OD600', 
                                            average_reps=['strain', 'plasmid', 'HU'])
    GC_AUC.append(gc_auc)
    GC_AUC_MEAN.append(gc_auc_mean.merge(strains_info, left_on='strain', right_on='strain'))

GC_AUC = pd.concat(GC_AUC).reset_index(drop=True)
GC_AUC = GC_AUC.loc[GC_AUC['strain']!='blank']

GC_AUC_MEAN = pd.concat(GC_AUC_MEAN).reset_index(drop=True)
GC_AUC_MEAN = GC_AUC_MEAN.loc[GC_AUC_MEAN['strain']!='blank']

for (ds, s, pl), df in GC_AUC.groupby(['dataset', 'strain', 'plasmid']):
    if 0 in df['HU'].values:
        GC_AUC.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['auc']).values

for (ds, s, pl), df in GC_AUC_MEAN.groupby(['dataset', 'strain', 'plasmid']):
    if 0 in df['HU'].values:
        GC_AUC_MEAN.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mean_auc']).values

# Growth curve fit to Logistic function

In [ ]:
GC_LOGISTIC = []
GC_LOGISTIC_MEAN = []

for od in ['OD600_blank', 'log_OD600']:
    if od == 'OD600_blank':
        bounds = (-np.inf, np.inf)
    elif od == 'log_OD600':
        bounds = ((0, 0, -np.inf), (np.inf, np.inf, np.inf))
    else:
        bounds = np.nan
        
    for ds, gc_dat in GC_DAT.groupby('dataset'):
        if experiments.loc[ds, 'blank'] == True:
            
            gc_logistic, gc_logistic_mean = mh.compute_gc_logistic(gc_dat, GC_samples[ds], strains_info, ds, 
                                                                   od=od, bounds=bounds,
                                                                   average_reps=['strain', 'plasmid', 'HU'])
            gc_logistic['OD'] = od
            gc_logistic_mean['OD'] = od
            
            GC_LOGISTIC.append(gc_logistic)
            GC_LOGISTIC_MEAN.append(gc_logistic_mean)

GC_LOGISTIC = pd.concat(GC_LOGISTIC).reset_index(drop=True)
GC_LOGISTIC_MEAN = pd.concat(GC_LOGISTIC_MEAN).reset_index(drop=True)

for (ds, s, pl, od), df in GC_LOGISTIC.groupby(['dataset', 'strain', 'plasmid', 'OD']):
    if 0 in df['HU'].values:
        GC_LOGISTIC.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mu']).values

for (ds, s, pl, od), df in GC_LOGISTIC_MEAN.groupby(['dataset', 'strain', 'plasmid', 'OD']):
    if 0 in df['HU'].values:
        GC_LOGISTIC_MEAN.loc[df.index, 'inhibition'] = mh.growth_to_inhibition(df.set_index('HU')['mean_mu']).values

GC_LOGISTIC_MEAN = GC_LOGISTIC_MEAN.merge(strains_info, left_on='strain', right_index=True)

# Fig 4 EF

In [ ]:
fig = plt.figure(figsize=[10, 3])

gs = plt.GridSpec(ncols=2, nrows=1, wspace=0.6, width_ratios=[4,3],
                 left=0.08, right=0.97, top=0.85, bottom=0.16)


ax = fig.add_subplot(gs[0])

dat = GC_LOGISTIC.loc[(GC_LOGISTIC['dataset']=='batch4') 
    & (GC_LOGISTIC['HU']==0)
    & (GC_LOGISTIC['plasmid'].isin(['empty', 'RAP1']))
    & (GC_LOGISTIC['OD']=='OD600_blank')]

sns.barplot(x='plasmid', y='mu', hue='background', 
             order=['empty', 'RAP1'],
             hue_order=strain_order, palette=strain_yprime_color,
             estimator='median', saturation=1, errorbar=None,
             data=dat, legend=False, ax=ax)

sns.stripplot(x='plasmid', y='mu', hue='background', 
              order=['empty', 'RAP1'],
              linewidth=0.5, edgecolor='w', size=3,
              hue_order=strain_order, dodge=True,
              palette='dark:k',
             data=dat, legend=False, ax=ax)

ax.set_ylim(0.15, 0.25)
ax.set_ylabel(r'OD$_{600}$ hr$^{-1}$ ')
ax.set_xlabel('')

legend_elms = [Line2D([0], [0], c='w', marker='s', ms=10,
                      label=s, mfc=strain_yprime_color[s]) for s in strain_order]
ax.legend(handles=legend_elms, loc=6, frameon=False, bbox_to_anchor=[1, 0.5])


# RAP1 plasmid

ax = fig.add_subplot(gs[1])

dat = dat.groupby(['strain','plasmid'])['mu'].median().reset_index()

dat1 = dat.set_index('plasmid').groupby('strain').apply(lambda x: x.loc['RAP1', 'mu']/x.loc['empty', 'mu'], include_groups=False).rename('ratio').reset_index()
dat1 = dat1.merge(strains_info, left_on='strain', right_index=True)
sns.scatterplot(x='yprime_kb', y='ratio', hue='background', 
                palette=strain_yprime_color, legend=False, 
                data=dat1, ax=ax)

lm = stats.linregress(dat1['yprime_kb'], dat1['ratio'])
X = np.array([0, 1400])
ax.plot(X, X*lm.slope+lm.intercept, c='0.5', lw=1)
ax.text(0.1, 0.95, 'r$^2$'+f'={lm.rvalue**2:.2f}\np={mh.plot_pval_text(lm.pvalue)}', 
        ha='left', va='top', c='0.5', transform=ax.transAxes)
ax.axhline(1, ls='--', lw=1, c='k', zorder=0)

ax.set_ylabel('RAP1/empty')
ax.set_xticks(np.linspace(0, 1500, 4))
ax.set_xlabel('Y\' kb')


sns.despine()

fig.text(0.01, 0.89, 'E', size=24, fontweight='bold')
fig.text(0.61, 0.89, 'F', size=24, fontweight='bold')

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig4EF.{ext}', dpi=300)

plt.show()
plt.close()